# Project 4 · Advanced solution — reusable validation descriptors

**Goal.** Implement `IntegerField` and `CharField` as data descriptors, then refactor common behavior into `BaseValidator`. This self-contained notebook covers both supplied *Solution — Part 1* and *Part 2*; no third-party runtime dependencies.

**Design decisions (deliberate improvements):** invalid value types raise `TypeError`, invalid ranges raise `ValueError`; booleans are not accepted as integers; bounds are checked during descriptor construction; integer bounds are inclusive; absent fields return `None` (as in the source material); negative minimum string lengths are clamped to zero, as in the source. Descriptors expect normal instances with `__dict__`; their per-instance state is stored in that dictionary. Python's underscore conventions and descriptors enforce the public API, not a security boundary against deliberately editing `__dict__`.

**Notebook execution:** run cells from top to bottom. Python 3.11+ is a convenient modern target, not the Python 3.6.7 version recorded in the provided notebook.

## 1 · Descriptor lifecycle and defensive validation

`__set_name__` runs automatically **when the owning class is created**. `__get__` returns the descriptor when accessed through the class and returns the instance's value otherwise. `__set__` validates before modifying storage; failed writes leave the prior value intact. This implementation explicitly rejects binding the **same descriptor object** to another attribute to avoid accidental shared binding metadata.

In [1]:
from __future__ import annotations
from numbers import Integral
from typing import Any


def _optional_int(value: int | None, name: str) -> None:
    """Validate a bound, deliberately excluding bool despite bool being an int subtype."""
    if value is not None and (isinstance(value, bool) or not isinstance(value, Integral)):
        raise TypeError(f"{name} must be an integer or None")


class BaseValidator:
    """Reusable data-descriptor protocol; subclasses specialize `validate`."""

    def __init__(self, min_: int | None = None, max_: int | None = None) -> None:
        _optional_int(min_, "min_")
        _optional_int(max_, "max_")
        if min_ is not None and max_ is not None and min_ > max_:
            raise ValueError("min_ must not exceed max_")
        self._min = min_
        self._max = max_
        self._owner: type | None = None
        self._name: str | None = None
        self._storage_name: str | None = None

    def __set_name__(self, owner: type, name: str) -> None:
        if self._owner is not None and (owner is not self._owner or name != self._name):
            raise ValueError("A descriptor instance cannot be bound to multiple attributes")
        self._owner = owner
        self._name = name
        self._storage_name = f"_validated_{name}"

    def _bound_storage(self) -> str:
        if self._storage_name is None:
            raise RuntimeError("Descriptor is unbound; create a class or call __set_name__")
        return self._storage_name

    def __get__(self, instance: object | None, owner: type | None = None) -> Any:
        if instance is None:
            return self
        return vars(instance).get(self._bound_storage(), None)

    def validate(self, value: Any) -> Any:
        """No-op validation hook; override in typed descriptors."""
        return value

    def __set__(self, instance: object, value: Any) -> None:
        storage = self._bound_storage()
        validated = self.validate(value)
        vars(instance)[storage] = validated

    def __delete__(self, instance: object) -> None:
        raise AttributeError(f"{self._name} cannot be deleted")


class IntegerField(BaseValidator):
    """Integer descriptor with optional inclusive bounds."""

    def validate(self, value: Any) -> Integral:
        name = self._name or "value"
        if isinstance(value, bool) or not isinstance(value, Integral):
            raise TypeError(f"{name} must be an integer (bool is not accepted)")
        if self._min is not None and value < self._min:
            raise ValueError(f"{name} must be >= {self._min}")
        if self._max is not None and value > self._max:
            raise ValueError(f"{name} must be <= {self._max}")
        return value


class CharField(BaseValidator):
    """String descriptor with optional inclusive length limits."""

    def __init__(self, min_: int | None = None, max_: int | None = None) -> None:
        _optional_int(min_, "min_")
        _optional_int(max_, "max_")
        normalized_min = max(0, min_ if min_ is not None else 0)
        if max_ is not None and max_ < 0:
            raise ValueError("max_ cannot be negative for string lengths")
        super().__init__(normalized_min, max_)

    def validate(self, value: Any) -> str:
        name = self._name or "value"
        if not isinstance(value, str):
            raise TypeError(f"{name} must be a string")
        if len(value) < self._min:
            raise ValueError(f"{name} must contain at least {self._min} characters")
        if self._max is not None and len(value) > self._max:
            raise ValueError(f"{name} must contain at most {self._max} characters")
        return value

## 2 · Declarative classes, isolation, and dynamic class creation

Each instance has its own stored values. A class-level access such as `Person.age` returns the descriptor itself. Using `type(name, bases, namespace)` calls `__set_name__` during class creation, making it particularly useful for unit-test fixtures with different bounds.

In [2]:
class Person:
    age = IntegerField(0, 130)
    name = CharField(1, 60)


alice = Person()
bob = Person()
alice.age, alice.name = 29, "Alice"
bob.age, bob.name = 36, "Bob"
assert (alice.age, bob.age) == (29, 36)
assert Person.age is Person.__dict__["age"]
assert Person.name is Person.__dict__["name"]
print(f"Alice: {alice.name}, {alice.age}; Bob: {bob.name}, {bob.age}")


DynamicPerson = type("DynamicPerson", (), {"score": IntegerField(0, 100)})
dynamic = DynamicPerson()
dynamic.score = 100
assert dynamic.score == 100
print("Dynamic class descriptor:", DynamicPerson.score, "score:", dynamic.score)

Alice: Alice, 29; Bob: Bob, 36
Dynamic class descriptor: <__main__.IntegerField object at 0x000002865669D1D0> score: 100


## 3 · The subtle post-creation binding trap

After class creation, `Sample.attribute = IntegerField(...)` does **not** automatically run `__set_name__`. Prefer declaring fields in the class body or constructing a class with `type`. If runtime replacement is unavoidable, call `descriptor.__set_name__(Sample, "attribute")` explicitly; this example uses a fresh descriptor object.

In [3]:
class LegacyRecord:
    pass

new_field = IntegerField(0, 10)
LegacyRecord.rating = new_field
new_field.__set_name__(LegacyRecord, "rating")
legacy = LegacyRecord()
legacy.rating = 8
assert legacy.rating == 8
print("Explicit late binding works:", legacy.rating)

Explicit late binding works: 8


## 4 · Automated tests — standard-library `unittest`

Tests cover normal and boundary values, unset fields, cross-instance isolation, inheritance, constructor errors, failed-write atomicity, class access, dynamic creation, and late binding. Each dynamic test class gets its **own new descriptor**. No intentionally failing cells are left in the final notebook.

In [4]:
import unittest


class TestBaseValidator(unittest.TestCase):
    def test_unvalidated_base_is_usable(self):
        class Record:
            item = BaseValidator()
        record = Record()
        self.assertIsNone(record.item)
        record.item = {"k": 1}
        self.assertEqual(record.item, {"k": 1})

    def test_class_attribute_returns_descriptor(self):
        self.assertIs(Person.age, Person.__dict__["age"])
        self.assertIs(Person.name, Person.__dict__["name"])

    def test_cross_instance_isolation(self):
        a, b = Person(), Person()
        a.age = 22
        self.assertIsNone(b.age)
        b.age = 35
        self.assertEqual((a.age, b.age), (22, 35))

    def test_inheritance(self):
        class Employee(Person):
            title = CharField(1, 20)
        employee = Employee()
        employee.age, employee.name, employee.title = 40, "Mia", "Engineer"
        self.assertEqual((employee.age, employee.name, employee.title), (40, "Mia", "Engineer"))

    def test_deletion_is_disallowed(self):
        person = Person()
        person.age = 20
        with self.assertRaises(AttributeError):
            del person.age
        self.assertEqual(person.age, 20)

    def test_rebinding_descriptor_rejected(self):
        field = IntegerField()
        class First:
            one = field
        with self.assertRaises(ValueError):
            field.__set_name__(First, "two")
        self.assertIs(First.one, field)

    def test_post_creation_explicit_binding(self):
        class Model:
            pass
        descriptor = IntegerField(0, 3)
        Model.score = descriptor
        descriptor.__set_name__(Model, "score")
        value = Model()
        value.score = 2
        self.assertEqual(value.score, 2)

    def test_unbound_descriptor_rejects_instance_write(self):
        descriptor = IntegerField()
        with self.assertRaises(RuntimeError):
            descriptor.__set__(object(), 1)


class TestIntegerField(unittest.TestCase):
    @staticmethod
    def make(min_=None, max_=None):
        return type("IntegerTestRecord", (), {"age": IntegerField(min_, max_)})()

    def test_inclusive_boundaries(self):
        p = self.make(0, 10)
        p.age = 0
        self.assertEqual(p.age, 0)
        p.age = 10
        self.assertEqual(p.age, 10)

    def test_below_minimum(self):
        p = self.make(0, 10)
        with self.assertRaisesRegex(ValueError, "age.*>= 0"):
            p.age = -1

    def test_above_maximum(self):
        p = self.make(0, 10)
        with self.assertRaisesRegex(ValueError, "age.*<= 10"):
            p.age = 11

    def test_minimum_only(self):
        p = self.make(0, None)
        p.age = 10000
        self.assertEqual(p.age, 10000)
        with self.assertRaises(ValueError):
            p.age = -1

    def test_maximum_only(self):
        p = self.make(None, 0)
        p.age = -10000
        self.assertEqual(p.age, -10000)
        with self.assertRaises(ValueError):
            p.age = 1

    def test_unbounded(self):
        p = self.make()
        for value in (-1000000, 0, 1000000):
            with self.subTest(value=value):
                p.age = value
                self.assertEqual(p.age, value)

    def test_bad_types_and_bool(self):
        p = self.make()
        for value in (True, False, 1.0, "1", None, 1+0j):
            with self.subTest(value=value):
                with self.assertRaises(TypeError):
                    p.age = value

    def test_invalid_bounds(self):
        for args, exc in (((5, 4), ValueError), ((1.0, 5), TypeError),
                          ((None, True), TypeError), ((False, None), TypeError)):
            with self.subTest(args=args):
                with self.assertRaises(exc):
                    IntegerField(*args)

    def test_invalid_write_preserves_previous(self):
        p = self.make(0, 10)
        p.age = 5
        with self.assertRaises(ValueError):
            p.age = 11
        self.assertEqual(p.age, 5)

    def test_dynamic_type_initializes_set_name(self):
        instance = self.make(1, 2)
        instance.age = 1
        self.assertEqual(instance.age, 1)
        self.assertIs(type(instance).age, type(instance).__dict__["age"])


class TestCharField(unittest.TestCase):
    @staticmethod
    def make(min_=None, max_=None):
        return type("CharacterTestRecord", (), {"name": CharField(min_, max_)})()

    def test_inclusive_length_boundaries(self):
        p = self.make(1, 3)
        for value in ("a", "abc"):
            with self.subTest(value=value):
                p.name = value
                self.assertEqual(p.name, value)

    def test_too_short_and_too_long(self):
        p = self.make(2, 4)
        for value in ("a", "abcde"):
            with self.subTest(value=value):
                with self.assertRaises(ValueError):
                    p.name = value

    def test_none_or_negative_minimum_clamps_to_zero(self):
        for minimum in (None, -10, 0):
            with self.subTest(minimum=minimum):
                p = self.make(minimum, None)
                self.assertEqual(type(p).name._min, 0)
                p.name = ""
                self.assertEqual(p.name, "")

    def test_maximum_only(self):
        p = self.make(None, 2)
        p.name = ""
        p.name = "ab"
        with self.assertRaises(ValueError):
            p.name = "abc"

    def test_unbounded(self):
        p = self.make()
        p.name = "x" * 1000
        self.assertEqual(len(p.name), 1000)

    def test_type_mismatch(self):
        p = self.make()
        for value in (None, True, 123, [], b"bytes"):
            with self.subTest(value=value):
                with self.assertRaises(TypeError):
                    p.name = value

    def test_invalid_bounds(self):
        for args, exc in (((10, 5), ValueError), ((None, -1), ValueError),
                          (("1", 10), TypeError), ((0, 2.5), TypeError)):
            with self.subTest(args=args):
                with self.assertRaises(exc):
                    CharField(*args)

    def test_invalid_write_does_not_erase_old_value(self):
        p = self.make(1, 3)
        p.name = "abc"
        with self.assertRaises(ValueError):
            p.name = ""
        self.assertEqual(p.name, "abc")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBaseValidator)
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestIntegerField))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestCharField))
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), "Advanced solution test failures"
print(f"Advanced solution: {result.testsRun} automated tests passed.")

..........................
----------------------------------------------------------------------
Ran 26 tests in 0.031s

OK


Advanced solution: 26 automated tests passed.


## 5 · Review and extension points

- A data descriptor (`__get__` + `__set__`) takes precedence over an instance dictionary entry **of the same name**; implementation stores values under private conventional keys. Access to raw `__dict__` bypasses validation and is not a supported mutation path.
- `None` means *not assigned* for this exercise; the extended notebook distinguishes required, optional, and defaulted model fields.
- Reusing the *same* descriptor instance for multiple names is prohibited here; use one descriptor object per field.
- Unit tests deliberately check `0` bounds, which rules out buggy truthiness checks such as `if self._min:`.
- For reusable code outside notebooks, move descriptors into a normal `.py` module and run the same tests with `python -m unittest` or pytest.